In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
if not os.path.exists('/content/work/data/vessel_catalog.csv'):
    os.system('mkdir -p /content/work/data /content/work/model /content/work/output /content/raw /content/baseline')
    os.system('unzip -o -q /content/drive/MyDrive/vessel/open_data.zip -d /content/raw')
    os.system('unzip -o -q /content/raw/open.zip -d /content/work/data')
    os.system('unzip -o -q /content/raw/baseline_submit.zip -d /content/baseline')
print(os.listdir('/content/work/data'))

In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)

D = '/content/work/data'
train = pd.read_csv(f'{D}/train.csv')
cat   = pd.read_csv(f'{D}/vessel_catalog.csv')
sub   = pd.read_csv(f'{D}/sample_submission.csv')

print(train.shape, cat.shape, sub.shape)

In [ ]:
print('== train 컬럼')
print(train.dtypes)
print('\n== catalog 컬럼')
print(cat.dtypes)

display(train.head(10))
display(cat.head(10))

In [ ]:
m = train.merge(cat, left_on='target_vessel_id', right_on='vessel_id', suffixes=('_rec', '_cat'))

cols = ['vessel_name_rec', 'vessel_name_cat',
        'gross_tonnage_rec', 'gross_tonnage_cat',
        'flag_name_rec', 'flag_name_cat',
        'vessel_type_rec', 'vessel_type_cat']
print(m[cols].sample(20, random_state=0).to_string())

# 데이터 검증셋 분할(선박 단위,고정)


In [ ]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_idx, va_idx = next(gss.split(train, groups=train['target_vessel_id']))
tr, va = train.iloc[tr_idx], train.iloc[va_idx]

print('학습용:', len(tr), '/ 검증용:', len(va))
print('양쪽에 겹치는 선박 수:', len(set(tr['target_vessel_id']) & set(va['target_vessel_id'])))
print('검증용 중 호출부호로 찾을 수 있는 비율:', round(va['call_sign'].isin(set(tr['call_sign'])).mean(), 3))

is_kor = lambda s: s.astype(str).str.contains('[가-힣]')
print('한글 비율 - 학습용:', round(is_kor(tr['vessel_name']).mean(), 3),
      '/ 검증용:', round(is_kor(va['vessel_name']).mean(), 3))

# 여기 아래부턴 실험 진행 코드

In [ ]:
# 단어 사전 만드는 셀

from collections import Counter, defaultdict

id2name = dict(zip(cat['vessel_id'], cat['vessel_name']))

def ko_tokens(s):
    toks = str(s).split()
    # 마지막 단어 끝의 '호'(~號) 제거: '파이오니어호' -> '파이오니어'
    if toks and len(toks[-1]) > 1 and toks[-1].endswith('호'):
        toks[-1] = toks[-1][:-1]
    return toks

def en_tokens(s):
    return str(s).upper().split()

# 1) 학습용 한글 기록에서 단어 쌍 모으기
kor_tr = tr[is_kor(tr['vessel_name'])]
pair_count = Counter()
n_aligned = 0
for ko, vid in zip(kor_tr['vessel_name'], kor_tr['target_vessel_id']):
    k, e = ko_tokens(ko), en_tokens(id2name.get(vid, ''))
    if len(k) == len(e):
        n_aligned += 1
        pair_count.update(zip(k, e))
print(f'단어 수가 같아서 짝지을 수 있었던 기록: {n_aligned} / {len(kor_tr)}')

# 2) 한글 단어마다 가장 자주 짝지어진 영문 단어 하나를 고름
ko2en = defaultdict(Counter)
for (k, e), c in pair_count.items():
    ko2en[k][e] += c
word_dict = {k: cnt.most_common(1)[0][0] for k, cnt in ko2en.items()}
print('사전 단어 수:', len(word_dict))

print('\n자주 나온 쌍 상위 30개:')
for (k, e), c in pair_count.most_common(30):
    print(f'  {k} -> {e}  ({c}회)')

# 3) 검증용 한글 단어 중 사전에 있는 비율
kor_va = va[is_kor(va['vessel_name'])]
va_toks = [t for s in kor_va['vessel_name'] for t in ko_tokens(s)]
print('\n검증용 한글 단어 중 사전에 있는 비율:', round(sum(t in word_dict for t in va_toks) / len(va_toks), 3))

In [ ]:
!pip install -q rapidfuzz==3.14.3 Unidecode==1.4.0

import numpy as np
from rapidfuzz import process, fuzz
from unidecode import unidecode

def translate(name):
    out = []
    for t in ko_tokens(name):
        out.append(word_dict[t] if t in word_dict else unidecode(t).upper())
    return ' '.join(out)

cat_names = cat['vessel_name'].astype(str).str.upper().values
cat_gt    = cat['gross_tonnage'].values
cat_ids   = cat['vessel_id'].values
all_idx   = np.arange(len(cat))

def match(name, gt, tol=0.10):
    q = translate(name)
    idx = all_idx
    if pd.notna(gt):
        cand = np.where(np.abs(cat_gt - gt) <= gt * tol)[0]
        if len(cand) > 0:
            idx = cand
    best = process.extractOne(q, cat_names[idx], scorer=fuzz.ratio)
    return cat_ids[idx[best[2]]], q

kor_va = va[is_kor(va['vessel_name'])].copy()
gt_in = kor_va['gross_tonnage'].fillna(kor_va['international_gross_tonnage'])
res = [match(n, g) for n, g in zip(kor_va['vessel_name'], gt_in)]
kor_va['pred2'] = [r[0] for r in res]
kor_va['변환'] = [r[1] for r in res]

acc = (kor_va['pred2'] == kor_va['target_vessel_id']).mean()
print(f'한글 정확도: 베이스라인 0.374 → 사전 매칭 {acc:.3f}')

kor_va['정답_이름'] = kor_va['target_vessel_id'].map(id2name)
print(kor_va[['vessel_name', '변환', '정답_이름']].head(20).to_string())

In [ ]:
gt_intl = kor_va['international_gross_tonnage'].fillna(kor_va['gross_tonnage'])
kor_va['pred3'] = [match(n, g, 0.10)[0] for n, g in zip(kor_va['vessel_name'], gt_intl)]
kor_va['hit3'] = kor_va['pred3'] == kor_va['target_vessel_id']

# 가설 1: 같은 배 중복
print('검증 한글 기록 수:', len(kor_va), '/ 고유 선박 수:', kor_va['target_vessel_id'].nunique())
print('한글 정확도 (기록 기준):', round(kor_va['hit3'].mean(), 3))
print('한글 정확도 (선박 기준, 중복 제거):', round(kor_va.drop_duplicates('target_vessel_id')['hit3'].mean(), 3))

# 가설 2: 카탈로그 톤수 결측
used = set(train['target_vessel_id'])
print('\n카탈로그 전체 톤수 결측 비율:', round(cat['gross_tonnage'].isna().mean(), 3))
print('train에 나온 배의 카탈로그 톤수 결측 비율:', round(cat.loc[cat['vessel_id'].isin(used), 'gross_tonnage'].isna().mean(), 3))
print('train에 안 나온 배의 카탈로그 톤수 결측 비율:', round(cat.loc[~cat['vessel_id'].isin(used), 'gross_tonnage'].isna().mean(), 3))

In [ ]:
# 1) baseline_lib.py에 어떤 함수들이 있는지
!grep -n "def \|class " /content/baseline/model/baseline_lib.py

# 2) 내 매칭 방식을 비한글 검증 기록에 적용
eng_va = va[~is_kor(va['vessel_name'])].copy()
g_en = eng_va['international_gross_tonnage'].fillna(eng_va['gross_tonnage'])
eng_va['pred_my'] = [match(n, g, 0.10)[0] for n, g in zip(eng_va['vessel_name'], g_en)]
print('\n비한글 정확도 - 베이스라인(rf): 0.989 / 내 매칭:', round((eng_va['pred_my'] == eng_va['target_vessel_id']).mean(), 3))

#submit 파일 생성


In [ ]:
import json

!rm -rf /content/submit_v5 && mkdir -p /content/submit_v5/model

kor_all = train[is_kor(train['vessel_name'])]
pc = Counter()
for ko, vid in zip(kor_all['vessel_name'], kor_all['target_vessel_id']):
    k, e = ko_tokens(ko), en_tokens(id2name.get(vid, ''))
    if len(k) == len(e):
        pc.update(zip(k, e))

k2e = defaultdict(Counter)
for (k, e), c in pc.items():
    k2e[k][e] += c
word_dict_full = {k: cnt.most_common(1)[0][0] for k, cnt in k2e.items()}

with open('/content/submit_v5/model/word_dict.json', 'w', encoding='utf-8') as f:
    json.dump(word_dict_full, f, ensure_ascii=False)

with open('/content/submit_v5/requirements.txt', 'w') as f:
    f.write('# Uses the evaluation image\'s preinstalled numpy, pandas, rapidfuzz and Unidecode.\n')

print('전체 사전 단어 수:', len(word_dict_full))

In [ ]:
%%writefile /content/submit_v5/model/matcher.py
"""Vessel matcher: name normalization (Korean dictionary + romanization) + tonnage filter + fuzzy match."""

import json
import re

import numpy as np
import pandas as pd
from rapidfuzz import process, fuzz
from unidecode import unidecode

HANGUL = re.compile("[가-힣]")


def is_korean(name):
    return bool(HANGUL.search(str(name)))


def ko_tokens(s):
    toks = str(s).split()
    if toks and len(toks[-1]) > 1 and toks[-1].endswith("호"):
        toks[-1] = toks[-1][:-1]
    return toks


class VesselMatcher:
    def __init__(self, catalog, dict_path, tol=0.10):
        with open(dict_path, encoding="utf-8") as f:
            self.word_dict = json.load(f)
        self.names = catalog["vessel_name"].astype(str).str.upper().values
        self.gt = catalog["gross_tonnage"].values
        self.ids = catalog["vessel_id"].values
        self.all_idx = np.arange(len(catalog))
        self.tol = tol

    def translate(self, name):
        if not is_korean(name):
            return str(name).upper()
        out = []
        for t in ko_tokens(name):
            out.append(self.word_dict.get(t, unidecode(t).upper()))
        return " ".join(out)

    def predict_one(self, name, gt):
        q = self.translate(name)
        idx = self.all_idx
        if pd.notna(gt):
            cand = np.where(np.abs(self.gt - gt) <= gt * self.tol)[0]
            if len(cand) > 0:
                idx = cand
        best = process.extractOne(q, self.names[idx], scorer=fuzz.ratio)
        return self.ids[idx[best[2]]]

In [ ]:
%%writefile /content/submit_v5/script.py
"""Vessel matching inference: one matcher for both Korean and non-Korean names."""

from pathlib import Path
import sys

import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parent / "model"))
from matcher import VesselMatcher


def run(
    test_path="data/test.csv",
    catalog_path="data/vessel_catalog.csv",
    dict_path="model/word_dict.json",
    output_path="output/submission.csv",
):
    test = pd.read_csv(test_path, encoding="utf-8-sig")
    catalog = pd.read_csv(catalog_path, encoding="utf-8-sig")
    if test["sample_id"].isna().any() or not test["sample_id"].is_unique:
        raise ValueError("test.csv sample_id must be non-null and unique")

    matcher = VesselMatcher(catalog, dict_path)

    # 카탈로그와 같은 국제총톤수를 우선 사용
    gt_in = test["international_gross_tonnage"].fillna(test["gross_tonnage"])
    predictions = [matcher.predict_one(n, g) for n, g in zip(test["vessel_name"], gt_in)]

    if len(predictions) != len(test) or not set(predictions).issubset(set(catalog["vessel_id"])):
        raise ValueError("Invalid catalog predictions")
    result = pd.DataFrame({"sample_id": test["sample_id"], "target_vessel_id": predictions})
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    result.to_csv(output_path, index=False, encoding="utf-8")
    print(f"Saved {len(result)} predictions to {output_path}")
    return output_path


if __name__ == "__main__":
    run()

In [ ]:
!rm -rf /content/test_run && cp -r /content/submit_v5 /content/test_run
!mkdir -p /content/test_run/data
!cp /content/work/data/vessel_catalog.csv /content/test_run/data/
va.drop(columns=['target_vessel_id']).to_csv('/content/test_run/data/test.csv', index=False)

!cd /content/test_run && time python script.py
!head -5 /content/test_run/output/submission.csv

In [ ]:
!cd /content/submit_v5 && rm -f /content/submit_5.zip && zip -r /content/submit_5.zip script.py requirements.txt model
!unzip -l /content/submit_5.zip

from google.colab import files
files.download('/content/submit_5.zip')